In [8]:
import os
import re
import subprocess
import tempfile

# Define paths
input_dir = r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level1_AudioGeneration\ExperimentAudio"
output_dir = r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level1_AudioGeneration\ExperimentAudio_2ch"
temp_dir = tempfile.mkdtemp()

# Create output directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

# Get list of all files in input directory
all_files = os.listdir(input_dir)

# Parse participant numbers and group files
participant_files = {}

for file in all_files:
    if file.endswith('.wav'):
        match = re.search(r'participant_([0-9]+)_design_(looming|tactile).wav', file)
        if match:
            participant_id = match.group(1)
            design_type = match.group(2)
            
            # Initialize dictionary entry if not exists
            if participant_id not in participant_files:
                participant_files[participant_id] = {}
                
            participant_files[participant_id][design_type] = file

print(f"Found files for {len(participant_files)} participants")

# Function to process audio files for each participant
def process_participant(participant_id, file_dict):
    # Skip if we don't have both looming and tactile files
    if 'looming' not in file_dict or 'tactile' not in file_dict:
        print(f"Skipping participant {participant_id} - missing files")
        return False
    
    looming_file = os.path.join(input_dir, file_dict['looming'])
    tactile_file = os.path.join(input_dir, file_dict['tactile'])
    output_file = os.path.join(output_dir, f"participant_{participant_id}_combined.wav")
    
    try:
        # Create a single FFmpeg command that does everything:
        # 1. Load both files
        # 2. Extract first channel from each
        # 3. Resample to 44100Hz
        # 4. Combine into stereo
        # 5. Output as WAV file
        
        cmd = [
            "ffmpeg",
            "-y",  # Overwrite output if exists
            "-i", looming_file,
            "-i", tactile_file,
            "-filter_complex", 
            "[0:a]channelsplit=channel_layout=stereo:channels=0,aresample=44100[left]; [1:a]channelsplit=channel_layout=stereo:channels=0,aresample=44100[right]; [left][right]amerge=inputs=2[aout]",
            "-map", "[aout]",
            "-c:a", "pcm_s16le",  # Standard PCM format
            "-ar", "44100",       # Sample rate 44100Hz
            "-ac", "2",           # Stereo output
            output_file
        ]
        
        print(f"Running FFmpeg...")
        result = subprocess.run(cmd, capture_output=True, text=True)
        
        if result.returncode != 0:
            print(f"FFmpeg error: {result.stderr}")
            
            # Try alternate approach with simplified filter
            print("Trying alternate approach...")
            cmd_alt = [
                "ffmpeg",
                "-y",
                "-i", looming_file,
                "-i", tactile_file,
                "-filter_complex", 
                "[0:a]pan=mono|c0=c0,aresample=44100[left]; [1:a]pan=mono|c0=c0,aresample=44100[right]; [left][right]join=inputs=2:channel_layout=stereo[aout]",
                "-map", "[aout]",
                "-c:a", "pcm_s16le",
                "-ar", "44100",
                output_file
            ]
            
            result_alt = subprocess.run(cmd_alt, capture_output=True, text=True)
            
            if result_alt.returncode != 0:
                # If that fails too, try an even simpler approach
                print("Trying simple approach...")
                cmd_simple = [
                    "ffmpeg",
                    "-y",
                    "-i", looming_file,
                    "-i", tactile_file,
                    "-filter_complex", 
                    "[0:a]aresample=44100[l];[1:a]aresample=44100[r];[l][r]amerge=inputs=2[aout]",
                    "-map", "[aout]",
                    "-ac", "2",
                    "-ar", "44100",
                    output_file
                ]
                result_simple = subprocess.run(cmd_simple, capture_output=True, text=True)
                
                if result_simple.returncode != 0:
                    print(f"All FFmpeg approaches failed: {result_simple.stderr}")
                    return False
        
        print(f"Successfully processed participant {participant_id}")
        return True
        
    except Exception as e:
        print(f"Error processing participant {participant_id}: {str(e)}")
        return False

# Process all participants
success_count = 0
total_count = len(participant_files)
print(f"Processing {total_count} participants...")

for participant_id, file_dict in participant_files.items():
    print(f"\nProcessing participant {participant_id} ({success_count+1}/{total_count})")
    if process_participant(participant_id, file_dict):
        success_count += 1

print(f"\nProcessing complete. Successfully processed {success_count} out of {total_count} participants.")

# Check first output file
output_files = [f for f in os.listdir(output_dir) if f.endswith('.wav')]
if output_files:
    first_file = os.path.join(output_dir, output_files[0])
    print(f"\nFirst output file created: {first_file}")

Found files for 100 participants
Processing 100 participants...

Processing participant 00 (1/100)
Running FFmpeg...
FFmpeg error: ffmpeg version 6.1 Copyright (c) 2000-2023 the FFmpeg developers
  built with clang version 17.0.4
  configuration: --prefix=/d/bld/ffmpeg_1699729642246/_h_env/Library --cc=clang.exe --cxx=clang++.exe --nm=llvm-nm --ar=llvm-ar --disable-doc --disable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --ld=lld-link --target-os=win64 --enable-cross-compile --toolchain=msvc --host-cc=clang.exe --extra-libs=ucrt.lib --extra-libs=vcruntime.lib --extra-libs=oldnames.lib --strip=llvm-strip --disable-stripping --host-extralibs= --enable-gpl --enable-libx264 --enable-libx265 --enable-libaom --enable-libsvtav1 --enable-libxml2 --enable-pic --enable-shared --disable-static --enable-version3 --enable-zlib --enable-libopus --pkg-config=/d/bld/ffmpeg_1699729642246/_build_env/Library/b